# API逐筆／五檔錄存品質核對（2026-11-18）
唯讀查詢實際本機錄存；輸出僅摘要、不含憑證、帳務或原始報價。以CPython順序執行，不冒稱Jupyter kernel。資料持續增加，重跑計數會不同；新完整盤不足，沒有新的可成交獲利結論。

In [1]:
from pathlib import Path
import hashlib, json, sqlite3
from datetime import datetime
from zoneinfo import ZoneInfo
import numpy as np
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'server/fubon_bridge').is_dir())
source = root / 'server/.market-capture.sqlite3'
zone = ZoneInfo('Asia/Taipei')


In [2]:
with sqlite3.connect(f'file:{source}?mode=ro', uri=True) as db:
    db.execute('BEGIN')
    maximum = db.execute('SELECT COALESCE(MAX(id),0) FROM capture_frames').fetchone()[0]
    query = """SELECT channel,COUNT(*),SUM(valid=0),SUM(is_trial=1),SUM(duplicate=1),
        MIN(received_us),MAX(received_us) FROM capture_frames WHERE id<=? GROUP BY channel"""
    channel_rows = db.execute(query,(maximum,)).fetchall()
    audit = dict(db.execute('SELECT kind,COUNT(*) FROM capture_audit GROUP BY kind').fetchall())
    enabled = bool(db.execute('SELECT enabled FROM capture_settings').fetchone()[0])
    formal_prints = db.execute("""SELECT COALESCE(SUM(json_array_length(raw_json,'$.data.trades')),0)
        FROM capture_frames WHERE id<=? AND channel='trades' AND valid=1 AND is_trial=0
        AND duplicate=0 AND event_type='data'""",(maximum,)).fetchone()[0]
    books = db.execute("""SELECT (received_us-event_us)/1000.,
        json_extract(raw_json,'$.data.asks[0].price')-json_extract(raw_json,'$.data.bids[0].price')
        FROM capture_frames WHERE id<=? AND channel='books' AND valid=1 AND is_trial=0""",(maximum,)).fetchall()
    windows = db.execute('SELECT DISTINCT symbol,month,session,opening_day FROM capture_frames WHERE id<=?',(maximum,)).fetchall()
assert sum(r[1] for r in channel_rows)==maximum
start = min(r[5] for r in channel_rows); end = max(r[6] for r in channel_rows)
evidence = dict(maximumFrameId=maximum,enabled=enabled,formalTradePrints=formal_prints,
    channels=[dict(channel=r[0],frames=r[1],invalid=r[2],trial=r[3],duplicate=r[4]) for r in channel_rows],
    firstReceived=datetime.fromtimestamp(start/1e6,zone).isoformat(),lastReceived=datetime.fromtimestamp(end/1e6,zone).isoformat(),
    windows=windows,audits=audit,bookLatencyMsQuantiles=np.quantile(np.asarray(books)[:,0],[.5,.95,.99]).tolist(),
    spreadPointsQuantiles=np.quantile(np.asarray(books)[:,1],[.5,.95,.99]).tolist())
print(json.dumps(evidence,ensure_ascii=False,indent=2))


{
  "maximumFrameId": 37230,
  "enabled": true,
  "formalTradePrints": 17517,
  "channels": [
    {
      "channel": "books",
      "frames": 21077,
      "invalid": 0,
      "trial": 0,
      "duplicate": 0
    },
    {
      "channel": "trades",
      "frames": 16153,
      "invalid": 0,
      "trial": 0,
      "duplicate": 0
    }
  ],
  "firstReceived": "2026-11-18T15:33:02.270593+08:00",
  "lastReceived": "2026-11-18T16:20:31.293539+08:00",
  "windows": [
    [
      "TMFL6",
      "202612",
      "afterhours",
      "2026-11-18"
    ]
  ],
  "audits": {
    "enabled": 1,
    "heartbeat": 95,
    "pong": 95,
    "subscriptions": 96,
    "window_change": 1
  },
  "bookLatencyMsQuantiles": [
    17.498,
    107.15700000000007,
    378.21956000000057
  ],
  "spreadPointsQuantiles": [
    4.0,
    5.0,
    6.0
  ]
}


In [3]:
status = json.loads((root/'server/.market-capture/status.json').read_text())
frozen_path = next(p for p in (root/'server/.market-capture/runs').glob('*/frozen.json') if json.loads(p.read_text())['id']==status['frozenId'])
frozen = json.loads(frozen_path.read_text())
for name, expected in frozen['implementation'].items():
    file = root/'docs/research/tmf-capture-protocol-v1.md' if name.endswith('.md') else root/'server/fubon_bridge'/name
    assert hashlib.sha256(file.read_bytes()).hexdigest()==expected, name
research = dict(state=status['state'],frozenId=status['frozenId'],
    sessions={k:{field:v[field] for field in ('observedWindows','coverageWindows','usableDays','fitDays')} for k,v in status['sessions'].items()},
    executionAllowed=status['executionAllowed'],adoptionAllowed=status['adoptionAllowed'],
    implementationFiles=len(frozen['implementation']))
assert not research['executionAllowed'] and not research['adoptionAllowed']
assert all(v['usableDays']==0 for v in research['sessions'].values())
print(json.dumps(research,ensure_ascii=False,indent=2))


{
  "state": "insufficient",
  "frozenId": "49ae6dc1d5391a02df39f12b6ededea12b5365cbc130ef8378db9be9937656cf",
  "sessions": {
    "regular": {
      "observedWindows": 0,
      "coverageWindows": 0,
      "usableDays": 0,
      "fitDays": 0
    },
    "afterhours": {
      "observedWindows": 1,
      "coverageWindows": 0,
      "usableDays": 0,
      "fitDays": 0
    }
  },
  "executionAllowed": false,
  "adoptionAllowed": false,
  "implementationFiles": 66
}
